# Table 3.   Factors associated with persisting untreated Pain, Multivariate Analysis

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/data/rega_data/trauma_categories_Rega Pain Study15.09.2025_v2.xlsx'
medic_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/data/rega_data/rega_physician_list_09102025.xlsx'
meta_medic_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/data/medreg_extraction/joined_final_complete_extractions_20251008_221735.xlsx'

In [ ]:
restrict_to_trauma = True
restrict_to_primary = True

In [ ]:
# Load and preprocess data
data_df = pd.read_excel(data_path)
medic_df = pd.read_excel(medic_data_path)
meta_medic_df = pd.read_excel(meta_medic_data_path)

# Merge with physician data
medic_df['full_name'] = medic_df['Mitglieder mit Einsatzfunktion'].str.replace(' (Flugarzt/Flugärztin)', '')
medic_df.drop_duplicates(subset=['Mitglieder mit Einsatzfunktion'], inplace=True)
medic_df = medic_df.merge(meta_medic_df, how='left', on='full_name')
medic_df.rename(columns={'Sex m/w': 'physician_sex'}, inplace=True)
data_df = data_df.merge(medic_df, how='left', left_on='Mitglieder mit Einsatzfunktion', right_on='Mitglieder mit Einsatzfunktion')

# Remove duplicates
data_df = data_df.drop_duplicates(subset=["SNZ Ereignis Nr. "])

# Filter patients with VAS > 3 at scene
data_df = data_df[data_df["VAS_on_scene"] > 3]

print(f"Total patients after filtering: {len(data_df)}")
print(f"Adult patients: {(data_df['Alter '] >= 16).sum()}")
print(f"Pediatric patients: {(data_df['Alter '] < 16).sum()}")

In [ ]:
if restrict_to_trauma:
    n_non_trauma = data_df[data_df['Einteilung (reduziert)'] != 'Unfall'].shape[0]
    print(f'Excluded {n_non_trauma} non-trauma patients')

    # adult non-trauma patients
    n_adult_non_trauma = data_df[(data_df['Einteilung (reduziert)'] != 'Unfall') & (data_df["Alter "] >= 16)].shape[0]
    print(f'Excluded {n_adult_non_trauma} adult non-trauma patients')
    # pediatric non-trauma patients
    n_pediatric_non_trauma = data_df[(data_df['Einteilung (reduziert)'] != 'Unfall') & (data_df["Alter "] < 16)].shape[0]
    print(f'Excluded {n_pediatric_non_trauma} pediatric non-trauma patients')

    data_df = data_df[data_df['Einteilung (reduziert)'] == 'Unfall']

In [ ]:
if restrict_to_primary:
    n_secondary = data_df[data_df['Einsatzart'] != 'Primär'].shape[0]
    print(f'Excluded {n_secondary} secondary transport patients')

    # adult secondary transport patients
    n_adult_secondary = data_df[(data_df['Einsatzart'] != 'Primär') & (data_df["Alter "] >= 16)].shape[0]
    print(f'Excluded {n_adult_secondary} adult secondary transport patients')
    # pediatric secondary transport patients
    n_pediatric_secondary = data_df[(data_df['Einsatzart'] != 'Primär') & (data_df["Alter "] < 16)].shape[0]
    print(f'Excluded {n_pediatric_secondary} pediatric secondary transport patients')
    data_df = data_df[data_df['Einsatzart'] == 'Primär']

In [ ]:
# Univariate logistic regression analysis
def univariate_logistic_regression(df, outcome_var, predictor_vars):
    """
    Perform univariate logistic regression for each predictor variable
    """
    results = []
    
    for var in predictor_vars:
        # Prepare data
        X = df[[var]].copy()
        y = df[outcome_var]
        
        # Add constant for intercept
        X_with_const = sm.add_constant(X)
        
        # Fit logistic regression
        try:
            model = sm.Logit(y, X_with_const).fit(disp=0)
            
            # Extract results
            coef = model.params[var]
            or_value = np.exp(coef)
            ci_lower = np.exp(model.conf_int().loc[var, 0])
            ci_upper = np.exp(model.conf_int().loc[var, 1])
            p_value = model.pvalues[var]
            
            results.append({
                'Variable': var,
                'Coefficient': coef,
                'OR': or_value,
                'CI_lower': ci_lower,
                'CI_upper': ci_upper,
                'P_value': p_value,
                'OR_CI': f"{or_value:.2f} ({ci_lower:.2f}-{ci_upper:.2f})",
                'P_formatted': f"{p_value:.3f}" if p_value >= 0.001 else "<0.001"
            })
            
        except Exception as e:
            print(f"Error with variable {var}: {e}")
            
    return pd.DataFrame(results)

# Adult Patients (≥16 years)

In [ ]:
# Prepare adult-only dataset (≥16 years)
from utils.utils import _extract_venous_access_features


adult_df = data_df[data_df['Alter '] >= 16].copy()

# insufficient pain management (VAS on arrival > 3)
adult_df['insufficient_pain_mgmt'] = (adult_df['VAS_on_arrival'] > 3).astype(int)

# drop na in adult_df['VAS_on_arrival']
adult_df = adult_df.dropna(subset=['VAS_on_arrival'])

# Create predictor variables for adult patients (exclude pediatric variable)
adult_df['age'] = adult_df['Alter ']
adult_df['male_patient'] = (adult_df['Geschlecht'] == 'Männlich').astype(int)
adult_df['male_physician'] = (adult_df['physician_sex'] == 'm').astype(int)

# get only year of event
adult_df['event_year'] = pd.to_datetime(adult_df['Ereignisdatum'], format='%d.%m.%Y').dt.year
adult_df['physician_age'] = adult_df['event_year'] - adult_df['year_of_birth']
# physician year of final exam (from licence_date which can be either d.m.Y or Y)
adult_df['physician_licence_year'] = adult_df['licence_date'].apply(lambda x: str(x).split('.')[-1] if '.' in str(x) else str(x))
adult_df['physician_experience_years'] = adult_df['event_year'] - pd.to_numeric(adult_df['physician_licence_year'], errors='coerce')

adult_df['physician_anesthesiologist'] = adult_df['specialist_qualifications'].str.contains('Anaesthesiology', na=False).astype(int)
adult_df['physician_intensivist'] = adult_df['specialist_qualifications'].str.contains('Intensive care medicine', na=False).astype(int)
adult_df['physician_internist'] = adult_df['specialist_qualifications'].str.contains('General Internal Medicine|General medical practitioner', na=False).astype(int)

adult_df['primary_mission'] = (adult_df['Einsatzart'] == 'Primär').astype(int)
adult_df['night_mission'] = (adult_df['Tag oder Nacht'] == 'Nacht').astype(int)
# separate into winter half-year (October to March) and summer half-year (April to September)
adult_df['winter_season'] = np.where(adult_df['Monat'].isin(['Oktober', 'November', 'Dezember', 'Januar', 'Februar', 'März']), 1, 0).astype(int)
adult_df['trauma'] = adult_df['Einteilung (reduziert)'].str.contains('Unfall', na=False).astype(int)
adult_df['winch_extraction'] = adult_df['Bergungen'].str.contains('Winde', na=False).astype(int)
adult_df['vas_scene'] = adult_df['VAS_on_scene']
adult_df['mission_duration'] = (
    pd.to_datetime(adult_df['Übergabezeit'], format='%d.%m.%Y %H:%M:%S') - 
    pd.to_datetime(adult_df['Erstbefund'], format='%d.%m.%Y %H:%M:%S')
).dt.total_seconds() / 60

# Create the same additional variables for adult patients
adult_df['short_scene_time'] = (adult_df['mission_duration'] < 30).astype(int)

venous_access_features = _extract_venous_access_features(adult_df["Zugänge"])
adult_df['venous_access_count'] = venous_access_features['venous_access_count']
adult_df['no_venous_access'] = (adult_df['venous_access_count'] == 0).astype(int)

# Create medication dose variables (matching Table 1 approach)
adult_df['fentanyl_dose'] = 0
adult_df['ketamine_dose'] = 0
adult_df['esketamine_dose'] = 0
adult_df['morphine_dose'] = 0
adult_df['Alle Medikamente'] = adult_df['Alle Medikamente'].str.replace(',', ';')  # replace commas with semicolons for consistency
for i, row in adult_df.iterrows():
    if pd.isna(row['Alle Medikamente']) or row['Alle Medikamente'] == 0:
        continue
    for analgetic in row['Alle Medikamente'].split(';'):
        if analgetic.strip() == '':
            continue
        # remove mcg or mg from dose
        if '7IE' in analgetic:
                print(f"Skipping dose with 7IE: {analgetic}")
                continue

        analgetic = analgetic.replace('mcg', '').replace('mg', '').strip()
        if 'Fentanyl' in analgetic and '/h' not in analgetic:
            dose = analgetic.split('Fentanyl')[-1].strip()
            adult_df.at[i, 'fentanyl_dose'] += float(dose) 
        elif 'Fentanyl' in analgetic and '/h' in analgetic:
            dose = analgetic.split('Fentanyl')[-1].strip().replace('/h', '')
            dose = float(dose) * adult_df.at[i, 'mission_duration']  
            adult_df.at[i, 'fentanyl_dose'] += float(dose)
        elif 'Ketamin' in analgetic or 'Ketamine' in analgetic:
            dose = analgetic.split('Ketamin')[-1].strip()
            adult_df.at[i, 'ketamine_dose'] += float(dose)
        elif 'Esketamin' in analgetic:
            dose = analgetic.split('Esketamin')[-1].strip()
            adult_df.at[i, 'esketamine_dose'] += float(dose)
        elif 'Morphin' in analgetic or 'Morphine' in analgetic:
            dose = analgetic.split('Morphin')[-1].strip()
            adult_df.at[i, 'morphine_dose'] += float(dose)

# fentanyl given
adult_df['fentanyl_given'] = adult_df['fentanyl_dose'] > 0
adult_df['morphine_given'] = adult_df['morphine_dose'] > 0
adult_df['ketamine_given'] = adult_df['ketamine_dose'] > 0
adult_df['esketamine_given'] = adult_df['esketamine_dose'] > 0

# Create combined medication variables
adult_df['any_opiate_dose'] = adult_df['morphine_dose'] + adult_df['fentanyl_dose']
adult_df['any_opiate_given'] = (adult_df['morphine_dose'] > 0) | (adult_df['fentanyl_dose'] > 0)
adult_df['any_ketamine_given'] = (adult_df['ketamine_dose'] > 0) | (adult_df['esketamine_dose'] > 0)

# combination therapy (opiate + ketamine) vs monotherapy
adult_df['opiate_ketamine_combination_therapy'] = (adult_df['any_opiate_given'] & adult_df['any_ketamine_given']).astype(int)

# Create no_analgesic variable: patients who receive no analgesic
# (no_analgesic corresponds to a dose of any_opiate and any_ketamine of 0)
adult_df['no_analgesic'] = ((adult_df['any_opiate_given'] == 0) & (adult_df['any_ketamine_given'] == 0)).astype(int)

# create outcome variable: insufficient pain management (VAS on arrival > 3) and no analgesic given
adult_df['persisting_untreated_pain'] = ((adult_df['insufficient_pain_mgmt'] == 1) & (adult_df['no_analgesic'] == 1)).astype(int)

In [ ]:
# get number of no analgesia with bootstrapping - example 32.2% (1663/5168; 95% CI 30.9-33.3%)
adult_df['no_analgesic'].value_counts(normalize=False)


In [ ]:
# get counts and percentage of night missions, winch_extraction, winter_season depending on adult_df['persisting_untreated_pain']
night_mission_counts = adult_df.groupby('persisting_untreated_pain')['night_mission'].value_counts(normalize=True)
winch_extraction_counts = adult_df.groupby('persisting_untreated_pain')['winch_extraction'].value_counts(normalize=True)
winter_season_counts = adult_df.groupby('persisting_untreated_pain')['winter_season'].value_counts(normalize=True)

print("Night Mission Counts:")
print(night_mission_counts)
print("\nWinch Extraction Counts:")
print(winch_extraction_counts)
print("\nWinter Season Counts:")
print(winter_season_counts)

In [ ]:
# compute associateion with no_analgesic of no_venous access, short_scene_time, winch_extraction, night_mission, winter_season
# example output print OR 2.60, 95% 2.17-3.10, p<0.001
no_venous_access_temp_df = adult_df[['no_analgesic', 'no_venous_access']].dropna()
venous_access_results = univariate_logistic_regression(no_venous_access_temp_df, 'no_analgesic', ['no_venous_access'])
print('No venous access association with no analgesic:')
print(venous_access_results[['Variable', 'OR_CI', 'P_formatted']])

# winch extraction
winch_extraction_temp_df = adult_df[['no_analgesic', 'winch_extraction']].dropna()
winch_extraction_results = univariate_logistic_regression(winch_extraction_temp_df, 'no_analgesic', ['winch_extraction'])
print('Winch extraction association with no analgesic:')
print(winch_extraction_results[['Variable', 'OR_CI', 'P_formatted']])

# night mission
night_mission_temp_df = adult_df[['no_analgesic', 'night_mission']].dropna()
night_mission_results = univariate_logistic_regression(night_mission_temp_df, 'no_analgesic', ['night_mission'])
print('Night mission association with no analgesic:')
print(night_mission_results[['Variable', 'OR_CI', 'P_formatted']])

# winter season
winter_season_temp_df = adult_df[['no_analgesic', 'winter_season']].dropna()
winter_season_results = univariate_logistic_regression(winter_season_temp_df, 'no_analgesic', ['winter_season'])
print('Winter season association with no analgesic:')
print(winter_season_results[['Variable', 'OR_CI', 'P_formatted']])

# # short scene time
# short_scene_time_temp_df = adult_df[['no_analgesic', 'short_scene_time']].dropna()
# short_scene_time_results = univariate_logistic_regression(short_scene_time_temp_df, 'no_analgesic', ['short_scene_time'])
# print('Short scene time association with no analgesic:')
# print(short_scene_time_results[['Variable', 'OR_CI', 'P_formatted']])